# 03. DR5 measurement: the full chain from the public catalogue

Everything from the 16.4 GB KiDS-Legacy (DR5) shear catalogue to the xi+/- and COSEBIs products, as `scripts/dr5_chain.py` stages that **resume across sessions**:

| stage | what | output |
|---|---|---|
| `download` | 7.2 GB gz from the KiDS server (resumable), decompressed to the 16.4 GB fits on the **local** disk | `/content/dr5/KiDS_Legacy_NS_unblind_final.fits` |
| `extract` | one pass over the catalogue, six per-bin npz | `dr5_work/bin0..5.npz` (1.6 GB, Drive) |
| `measure` | TreeCorr `bin_slop` 1.5, one process per tomographic pair (21 pairs) | `runs/dr5_measure/xi_unordered/b1.5/Bin*_Bin*.npz` |
| `eval`, `product` | comparison with the official release, xi+/- and COSEBIs product, full-vector checks, step D | `runs/dr5_measure/product/` |
| `plots` | the `dr5_*` figures | `$KIDSPIPE_PLOTS` |
| `exact` | exact run (`bin_slop` 0, `angle_slop` 0, 16 chunks), 4656 chunk-pairs, chunk-level resumable | `runs/dr5_measure/xi_chunked/chunks/` (346 MB) |
| `assemble`, `exact-product` | assemble the chunks, exact product and its comparison with the `bin_slop` 1.5 product | `runs/dr5_measure/product_exact/` |

The science settings are those of the scripts (the chain only decides which pair runs when). The shipped repository already contains the reference results of this chain
(`runs/dr5_measure/`); your run goes to `$KIDSPIPE_RUNS` on Drive and `compare` shows the differences.

**Acknowledgement.** The DR5 / KiDS-Legacy data are KiDS products. Acknowledge KiDS and cite Wright et al. 2024 (A&A 686, A170) and the KiDS-Legacy papers
(<https://kids.strw.leidenuniv.nl/DR5/legacy_wl.php>); the full acknowledgement statement is on that KiDS page (the repository's `README.md` and `data/README.md` only summarise it). The repository is private; do not share the data or results
outside the terms of the KiDS release.

### Which runtime?

| | free Colab | Colab Pro / high-RAM |
|---|---|---|
| vCPU | 2 | about 8 |
| RAM | 12.7 GB | about 51 GB |
| session length | about 12 h (idle and background limits apply) | up to about 24 h |
| local disk | about 100 GB (wiped at the end of the session) | the same |
| Google Drive | persistent, shared 15 GB quota (free) | persistent |

The notebooks use `os.cpu_count()` for threads and worker processes. All times in these notebooks are **estimates** scaled from timings recorded on a
22-core development workstation (WSL2): linear scaling in cores is assumed and a Colab vCPU is assumed to be as fast as one of its cores, which is probably
optimistic. Treat them as lower bounds on a shared, noisy Colab host.

### Cost and sessions (ESTIMATES)

| | free, 2 vCPU | Pro, 8 vCPU |
|---|---|---|
| download + decompress + extract | 0.5-1.5 h (network and disk bound) | the same |
| `measure` (21 pairs, `bin_slop` 1.5) | about 63 h = 6 sessions of 11 h (cross pair 3.3-3.8 h, auto pair 1.6-1.9 h) | about 16 h = 2 sessions of 11 h or 1 of 24 h (cross 50-57 min, auto 24-28 min) |
| `eval`, `product`, `plots` (a4 re-check included) | about 30-60 min | about 15 min |
| `exact` (4656 chunk-pairs) | about 390 h = 36 sessions: **not practical**, use the shipped exact product as the reference | about 98 h = 9 sessions of 11 h (a chunk-pair near the diagonal about 22 min) |
| `assemble`, `exact-product` | minutes | minutes |

Recorded on the 22-core workstation with 20 threads: `measure` 6.32 h (autos 564-683 s, crosses 1170-1356 s), `exact` 39.3 h in three sessions (1.30-2.31 h per pair).
The estimates scale these by 1/threads. After the first pair finishes on your machine, `dr5_chain.py status` re-calibrates them from the measured time.

**Memory.** One tomographic pair per process (a process that held many pairs was killed by the OOM killer): a cross pair peaks at about 5 GB RSS, an auto pair at about 2.6 GB.
Fine on the free 12.7 GB runtime; nothing is gained from more RAM because pairs run one after the other, each with all threads.

**Drive space.** Per-bin npz 1.6 GB; `b1.5` pairs 1.7 MB; exact chunks 346 MB; products a few MB; the 7.2 GB gz only if you keep it on Drive (the notebook keeps it local and can delete it,
`--delete-gz`). Total about 2.2 GB of the free 15 GB Drive quota. **Never keep the 16.4 GB fits on Drive**: the FUSE mount is slow and the extraction reads all 16.4 GB.

**Sessions.** Every command is idempotent; the truth is read from the files. `--max-hours H` is the budget of one invocation: no pair is started whose estimated time exceeds what is left; the exact run
is stopped at the deadline (it loses at most the chunk-pair in progress). Exit status 75 means "not finished, run the same command in a new session". The state and timings are in
`MyDrive/kidspipe/runs/dr5_measure/chain_state.json`.

### One-time preparation: access to the private repository

The repository is private, so the notebook needs a GitHub **fine-grained personal access token**. The token is stored in **Colab Secrets**
and is never written into a cell, a file or the output.

1. GitHub, Settings, Developer settings, Personal access tokens, **Fine-grained tokens**, Generate new token. Resource owner: the account that owns the
   repository. Repository access: **Only select repositories**, choose this repository. Permissions: **Contents: Read-only** (Metadata is added
   automatically). Pick a short expiry. Copy the token once.
2. In Colab, left sidebar, the key icon (**Secrets**), Add new secret: name `GH_TOKEN`, value = the token, switch **Notebook access** on for this notebook
   (the switch is per notebook).
3. Edit `GITHUB_REPO` in the setup cell below (`owner/name`).

No GitHub access? Put a zip of the repository at `MyDrive/kidspipe/kidspipe_repo.zip`; the setup cell unpacks it instead of cloning.

## Session setup

A Colab runtime is fresh in every session: **run the next cell first, every time**. It mounts Drive, clones or updates the repository, installs the package if needed and sets the environment variables (`KIDSPIPE_RUNS`, `KIDSPIPE_DR5_WORK`, `KIDSPIPE_DR5_CATALOGUE`, `KIDSPIPE_PLOTS`, thread counts, numba cache on Drive) described in `docs/COLAB.md`.

In [ ]:
#@title 1. Session setup (run this first, in EVERY session) { display-mode: "form" }
#@markdown Private repository to clone (`owner/name`); the token comes from the Colab Secret `GH_TOKEN`.
GITHUB_REPO = "OWNER/REPO"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
#@markdown Persistent outputs live on Drive; the 16 GB catalogue lives on the local disk (never on Drive).
DRIVE_ROOT = "/content/drive/MyDrive/kidspipe"  #@param {type:"string"}
CATALOGUE_DIR = "/content/dr5"  #@param {type:"string"}
REPO_DIR = "/content/kidspipe"

import base64, importlib, os, pathlib, shutil, subprocess, sys

def _git(args, token=None, cwd=None):
    """git, with the token passed as an HTTP header (not stored in .git/config); the token is masked in anything returned."""
    cmd, secret = ["git"], []
    if token:
        b64 = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        cmd += ["-c", f"http.extraheader=Authorization: Basic {b64}"]
        secret = [token, b64]
    r = subprocess.run(cmd + args, cwd=cwd, capture_output=True, text=True)
    out = r.stdout + r.stderr
    for s in secret:
        out = out.replace(s, "***")
    return r.returncode, out.strip()

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab: Drive is not mounted; DRIVE_ROOT must be a writable directory.")

token = None
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception as e:                                   # secret missing, or notebook access not granted
    print(f"No GH_TOKEN secret available ({type(e).__name__}); using an existing clone or the Drive zip if there is one.")

repo = pathlib.Path(REPO_DIR)
url = f"https://github.com/{GITHUB_REPO}.git"
if (repo / "pyproject.toml").exists():
    rc, out = _git(["fetch", "--depth", "1", "origin", BRANCH], token, cwd=repo)
    if rc == 0:
        _git(["reset", "--hard", "FETCH_HEAD"], cwd=repo)
        print("repository updated to the latest", BRANCH)
    else:
        print("could not update (using the existing clone):", out[-300:])
elif token or not pathlib.Path(DRIVE_ROOT, "kidspipe_repo.zip").exists():
    if GITHUB_REPO.startswith("OWNER/"):
        raise SystemExit("Edit GITHUB_REPO (owner/name) in this cell, and add the GH_TOKEN secret (see the markdown above).")
    rc, out = _git(["clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR], token)
    if rc != 0:
        raise SystemExit("git clone failed (wrong GITHUB_REPO, or the token lacks Contents: read access?):\n" + out[-500:])
    print("cloned", GITHUB_REPO)
else:
    tmp = pathlib.Path("/content/_unzip"); shutil.rmtree(tmp, ignore_errors=True)
    shutil.unpack_archive(str(pathlib.Path(DRIVE_ROOT, "kidspipe_repo.zip")), str(tmp))
    top = [p for p in tmp.iterdir() if p.is_dir()]
    shutil.move(str(top[0]) if len(top) == 1 and not (tmp / "pyproject.toml").exists() else str(tmp), REPO_DIR)
    print("unpacked the repository from the Drive zip")

sys.path.insert(0, f"{REPO_DIR}/scripts/colab")
import setup_env; importlib.reload(setup_env)
INFO = setup_env.setup(REPO_DIR, DRIVE_ROOT, CATALOGUE_DIR)      # env vars, pip install -e '.[all]' if needed, versions
os.chdir(REPO_DIR)
WORKERS = os.cpu_count() or 1

In [ ]:
#@title Session parameters { display-mode: "form" }
MAX_HOURS = 11  #@param {type:"number"}
#@markdown free Colab: about 11 (the session ends at about 12 h); Pro: up to about 23
DELETE_GZ = True  #@param {type:"boolean"}
LOCAL_WORK_CACHE = True  #@param {type:"boolean"}
#@markdown copy the per-bin npz from Drive to the local disk once per session (faster than reading Drive in every pair process)
import os, pathlib
from setup_env import sh, launch, tail, stop
W = WORKERS
CHAIN = f"python3 scripts/dr5_chain.py"
CACHE = " --work-cache /content/dr5_work_local" if LOCAL_WORK_CACHE else ""
COMMON = f"--threads {W}"
LOG = pathlib.Path(os.environ["KIDSPIPE_RUNS"]) / "dr5_measure" / "chain_logs" / "session.log"
def run(sub, extra=""):
    rc = sh(f"{CHAIN} {sub} {COMMON} {extra}")
    print(f"\nexit status {rc}" + ("  -> not finished: run this cell again in a new session" if rc == 75 else "  (failed: see the output above)" if rc else "  (complete)"))
    return rc

## What is done and what is left

In [ ]:
run('status')

## Option A: one cell, re-run in every session

Runs every stage in order and stops cleanly when the budget `MAX_HOURS` is used up. Run it, wait, and when the session ends or the cell stops with status 75, open a new session,
run the setup cell and this cell again. Remove `--no-exact` to continue into the exact run (Pro / 8 vCPU).

In [ ]:
rc = run("all", f"--max-hours {MAX_HOURS}{CACHE} --no-exact" + (" --delete-gz" if DELETE_GZ else ""))

## Option B: stage by stage

### B1. Session A: download, extract (about 0.5-1.5 h)

In [ ]:
run("download", "--delete-gz" if DELETE_GZ else "")
run("extract")

### B2. Sessions B...: the `bin_slop` 1.5 measurement

Re-run this cell in every session until it reports "all requested pairs done". `--order heavy` (default) starts the long cross pairs first and fills the end of the budget with the shorter auto pairs.

In [ ]:
rc = run("measure", f"--max-hours {MAX_HOURS}{CACHE}")

Alternative for long runs: start the stage **detached** so that closing the browser tab does not stop it (it keeps running while the runtime lives), then poll the log.
Stop it with `stop(p)` (signals the whole process group; `p.terminate()` would only reach the shell wrapper; the chain terminates the running pair cleanly; finished pairs stay on disk).

In [ ]:
START_DETACHED = False   #@param {type:"boolean"}
if START_DETACHED:
    p = launch(f"{CHAIN} measure {COMMON} --max-hours {MAX_HOURS}{CACHE}", LOG)

In [ ]:
print(tail(LOG, 25))

### B3. Evaluate, make the product, plots (needs all 21 pairs)

`product` runs the release-only integration check, builds the xi+/- and COSEBIs product with the full-vector checks, and the model chi2 (step D). `plots` runs the estimator re-check on bins 1 and 6
(`a4`, a few minutes) and writes the `dr5_01...06` figures to `$KIDSPIPE_PLOTS`.

In [ ]:
run("eval")
run("product")
run("plots")

### B4. Compare with the shipped reference results

`compare` lists, for each pair file and each product, whether your arrays are bit-identical to the reference results shipped in the repository or how large the difference is
relative to the reference. Differences at the level of floating-point rounding are expected when TreeCorr, numpy or the BLAS differ from the development workstation; the agreement criteria of the
measurement are in `docs/DR5_MEASUREMENT_REPORT.md` and are not redefined here.

In [ ]:
run("compare")
from IPython.display import Image, display
for n in ("dr5_01_xipm_vs_official.png", "dr5_03_En_vs_official.png"):
    for label, d in (("yours", os.environ["KIDSPIPE_PLOTS"]), ("shipped", "/content/kidspipe/plots")):
        f = pathlib.Path(d) / n
        print(label, f, "(missing)" if not f.exists() else "")
        if f.exists(): display(Image(filename=str(f), width=700))

### B5. The exact run (Pro / 8 vCPU; chunk-level resumable)

One process per tomographic pair, exact settings (`bin_slop` 0, `angle_slop` 0, 16 chunks). At the deadline the running pair is stopped; at most the chunk-pair in progress is lost (it is recomputed next time).
Use `--no-kill` to only start pairs whose estimate fits. Re-run until it reports "all requested pairs complete". `--pairs 1-1,2-3` selects pairs.

In [ ]:
rc = run("exact", f"--max-hours {MAX_HOURS}{CACHE}")

In [ ]:
if rc == 0:
    run("assemble")
    run("exact-product")
    run("plots")
    run("compare")

### Archive the results

Everything persistent is already on Drive (`MyDrive/kidspipe/`). To take the products away:

In [ ]:
import shutil
arc = shutil.make_archive("/content/dr5_products", "zip", os.environ["KIDSPIPE_RUNS"], "dr5_measure/product") if (pathlib.Path(os.environ["KIDSPIPE_RUNS"]) / "dr5_measure/product").exists() else None
print(arc)

### Troubleshooting

* **`status` shows everything "done" immediately.** `KIDSPIPE_RUNS` points at the repository's shipped `runs/`, not at Drive: run the setup cell (it sets the Drive path).
* **A pair was killed (exit -9 / 137).** Out of memory. Make sure nothing else holds RAM (restart the runtime), and that only one pair runs. The chain never runs two pairs at once.
* **Download stops.** Run `download` again; it resumes (`curl -C -`). If the decompression fails the gz is truncated: delete it and download again.
* **Slow `measure`.** Check `status` for the machine factor; Colab vCPUs can be 2x slower than the workstation cores. Check `LOCAL_WORK_CACHE`.
* **The session was reclaimed.** Nothing is lost except the pair (or chunk-pair) in progress: new session, setup cell, same stage cell.